# Mask accurary will be measured here

> Mask accuracy

In [ ]:
#| default_exp mask_accuracy

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm import tqdm
from fastcore.all import *
import matplotlib as mpl
import matplotlib.pyplot as plt
from fastcore.all import *
import shutil
import cv2
from typing import Union, List, Tuple, Dict
import pandas as pd
from skimage import io, morphology, measure
from scipy.ndimage import (label, sum, binary_fill_holes)
import os
from tqdm.auto import tqdm
import argparse

In [ ]:
#| export
custom_lib_path = Path("/home/ai_warstein/homes/goni/custom_libs")
current_lib_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test')
sys.path.append(str(current_lib_path))
sys.path.append(str(custom_lib_path))
from dotenv import load_dotenv
load_dotenv(dotenv_path="/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/.env")

CV_TOOLS = os.getenv("CV_TOOLS")
P_EPD= os.getenv("PRIVATE_EASY_PIN_DETECTION")
sys.path.append(CV_TOOLS)
sys.path.append(P_EPD)

In [ ]:
from cv_tools.core import *

In [ ]:
#| hide
mpl.rcParams['image.cmap']='gray'

In [ ]:
import shutil
chunk_number = 4
chunks_path = Path(fr'/home/ai_easypid.work/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/patch_with_synthetic_fail_data/results/chunk_{chunk_number:04d}/predictions')
chunk_images = chunks_path.ls(file_exts='.png')
chunk_images

for i in range(124):
    data_path = Path(fr'/home/ai_easypid.work/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/patch_with_synthetic_fail_data_epoch_100_model/results/chunk_{i:04d}/predictions')
    shutil.rmtree(data_path)

In [ ]:
from new_test.core import *

In [ ]:
from new_test.model_eval.pass_fail_image_copy import create_df

In [ ]:
chunk_images[0].parent

In [ ]:
dfs =[]
wrong_df = []
for i in range(124):
    data_path = Path(fr'/home/ai_easypid.work/data/projects/easy_end_test/ET4/1B_solder/1B_solder_unzip/output/results/chunk_{i:04d}/predictions')
    try:
        df = create_df(mask_path=data_path)
        dfs.append(df)
    except Exception as e:
        print(e)
        wrong_df.append(data_path)
        
    

In [ ]:
wh_df = pd.concat(dfs)
wh_df.shape

In [ ]:
w_msk.shape

In [ ]:
c_msk = wh_df.query('diff ==0')
w_msk = wh_df.query('diff != 0')

In [ ]:
w_msk['file_path'].values[0]

In [ ]:
def process_image_im1_im2_j(
    im2:np.ndarray,
    im1:np.ndarray
   )->[Tuple[np.ndarray, np.ndarray]]: # return two image of shape (None, 2, 1152, 1632)
    'process image 1 and image2 to make it work with two image model'
    #if len(im1.shape) <3:
        #im1 = np.expand_dims(im1, axis=-1)
        #print(im1.shape)
    #if len(im2.shape) <3:
        #im2 = np.expand_dims(im2, axis=-1)
        #print(im2.shape)



    im1 = im1.astype(np.float32)/255.0
    im2 = im2.astype(np.float32)/255.0
    p_img = np.stack((im1, im2), axis=-1)[None,...]
    return torch.from_numpy(p_img).permute(0,3,1,2)

In [ ]:
im_root = Path(r'/home/ai_easypid.work/data/projects/easy_end_test/ET4/1B_solder/1B_solder_unzip')
im1_root = Path(im_root, 'main_im1_cropped')
im2_root = Path(im_root, 'main_im2_cropped')
im_name = 'VFV4.9.0.3_2025051214261248_In_17_Missing Lead_image2.png'
im1_fn=Path(im1_root, im_name.replace('image2', 'image1'))
im2_fn=Path(im2_root, im_name)
#show_(im1_fn)
im1_fn.exists(),im2_fn.exists()

In [ ]:
import torch

In [ ]:
im1_img = read_img(im1_fn)
im2_img = read_img(im2_fn)

In [ ]:
pr_img = process_image_im1_im2_j(im1_img, im2_img)

In [ ]:
pr_img.shape

In [ ]:
model_path = Path(im_root, 'output')

In [ ]:
# Test ONNX model for comparison
onnx_model_path = Path(model_path, 'flex_model.onnx')
if onnx_model_path.exists():
    import onnxruntime as ort
    
    # Load ONNX model
    ort_session = ort.InferenceSession(str(onnx_model_path))
    
    # Prepare input for ONNX (convert to numpy)
    onnx_input = comb_img.cpu().numpy()
    
    # Run ONNX inference
    onnx_outputs = ort_session.run(None, {'input': onnx_input})
    onnx_pred = torch.from_numpy(onnx_outputs[0])
    
    # Show ONNX model output
    show_(overlay_mask_border_on_image_frm_img(
        sm_img2_img,
        onnx_pred[0,0,:,:].numpy()*255
    ))
    

In [ ]:
c_msk['file_path'].values[0]

In [ ]:
show_(w_msk['file_path'].values[0])

In [ ]:
show_(c_msk['file_path'].values[0])

In [ ]:
dfs =[]
wrong_df = []
for i in range(124):
    data_path = Path(fr'/home/ai_easypid.work/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/patch_with_synthetic_fail_data_epoch_100_model/results/chunk_{i:04d}/predictions')
    try:
        df = create_df(mask_path=data_path)
        dfs.append(df)
    except Exception as e:
        print(e)
        wrong_df.append(data_path)
        
    

In [ ]:
whole_df = pd.concat(dfs)

In [ ]:
wrong_masks = whole_df.query('diff !=0')

In [ ]:
wrong_masks

In [ ]:
wrong_masks['file_path'].values[0]

In [ ]:
wrong_masks.to_parquet('cache/patching_additional_pin_wo_anything.parquet')

In [ ]:
show_(wrong_masks['file_path'].values[0])

In [ ]:
#| export
from new_test.core import *
from private_easy_pin_detection.extract_patches import *
from private_easy_pin_detection.pin_map import *
from cv_tools.core import *
from cv_tools.cv_ops import *

In [ ]:
r_empty = np.vectorize(lambda x: x.replace(' ', '_'))

In [ ]:
#| hide
model_name='time_20_30_50_val_frGrnd0.9389_epoch_174.h5'
im_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail_17_folder/Fail_17_cropped')
im_save_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail_17_folder/only_pin_pos_masks')
im_save_path.mkdir(exist_ok=True, parents=True)
msk_path = Path(fr'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail_17_folder/new_pin_full_image_model_masks/{model_name}')
tmp_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/templates/17.png')
msk_path.ls(), im_path.ls()

In [ ]:
img_names = get_name_(im_path.ls())
common_msks =[i for i in msk_path.ls() if i.name in r_empty(img_names)]

In [ ]:
Path(msk_path.ls()[0]).name

In [ ]:
img_fn = im_path.ls()[0]
sample_fn = r_empty(f'{get_name_(img_fn)}')
msk_fn = msk_path/f'{sample_fn}'
read_img(msk_fn,cv=False)

In [ ]:
#| export
def desired_im_folder(
        im_path:Path,#folder for images
        msk_path:Union[Path, None ],#folder for masks
        tmp_path:Path, #folder for templates
        recipe_name:str,
        img_sn_pin_save:Union[Path, str, None]=None, #folder for saving the single pin images 
        msk_sn_pin_save:Union[Path, str, None]=None, #folder for saving the single pin masks
        save_path:Union[Path, str, None]=None #folder for saving the last part
    ):
    'Save only desired positions part from the image or mask'
    
    mask_names = get_name_(msk_path.ls())
    im_names = list(filter(lambda x: x.name in mask_names, msk_path.ls()))

    print(f'Number of images: {len(im_names)}')


    for i in tqdm(im_names, total=len(im_names)):
        name_ = Path(i).name
        full_, _ = desired_im(
            im_path=Path(im_path, name_),
            msk_path=Path(msk_path, name_),
            tmp_path=tmp_path,
            recipe_name=recipe_name,
            img_sn_pin_save_path=img_sn_pin_save,
            msk_sn_pin_save_path=msk_sn_pin_save,
        )
        if save_path is not None:
            cv2.imwrite(str(Path(save_path, name_)), full_)





In [ ]:
desired_im_folder(
    im_path=im_path,
    msk_path=msk_path,
    tmp_path=tmp_path,
    recipe_name='17',
    save_path=im_save_path

)

In [ ]:
full_msk, prt_msk=desired_im(
    im_path=img_fn,
    tmp_path=tmp_path,
    msk_path=msk_fn,
    img_sn_pin_save_path=None,
    msk_sn_pin_save_path=None,
    recipe_name="18",
)

In [ ]:
#| export
def process_pin_count(
    df: pd.DataFrame,  # dataframe containing pin count and difference
):
    return (
        df.assign(
            pin_type=lambda df_: df_['device_name'].astype(str).fillna('Undefined').map(
                lambda x: get_pin_type().get(x, '0'), na_action='ignore'),
            pin_type_g=lambda df_: np.where(
                df_['pin_type'].astype(str).str.contains('pressfit'),
                'pressfit', 'solder'),
            expected=lambda df_: df_['device_name'].astype(str).fillna('Undefined').map(
                lambda x: pin_count_dict().get(x, '0'), na_action='ignore').astype(float),
            expected_vs_diff=lambda df_: df_['pin_no'] - df_['expected']
        )
    )

In [ ]:
#| export
def process_mask(
        msk_img:np.ndarray, # binary mask
        remove_object_size:80000, # object size
        fill_holes_: bool = True,
        fit_rect_: bool = True,
        show_msk: bool = False
        ):
    if remove_object_size is not None:
        new_mask = remove_small_objects(
            binary_mask=msk_img,
            size_threshold=remove_object_size)
    else:
        new_mask = msk_img
    if fill_holes_:
        new_mask = fill_holes_in_objects(new_mask)
    if fit_rect_:
        new_mask = convert_to_rotated_rectangles(binary_mask=new_mask)
    if show_msk:
        show_(new_mask)
    return new_mask

In [ ]:
#| hide
msk_path = Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/Fail18_masks')
msk_path.ls()

In [ ]:
#| export
def process_mask_folder(
        path:Union[str, Path], # path to mask folder
        remove_object_size:int=80000,
        fil_holes_:bool=True,
        fit_rect_:bool=True,
        save_path:Union[str, Path]=None,
     ):
    'Process(remove small object, fill_holes, fit_rect) all masks inside path'
    fns = Path(path).ls()
    Path(save_path).mkdir(exist_ok=True, parents=True)  
    for i in tqdm(fns, total=len(fns)):
        msk_img = read_img(f'{i}', cv=True, gray='True')
        name_= Path(i).name
        pr_msk = process_mask(
            msk_img=msk_img,
            remove_object_size=remove_object_size,
            fill_holes_=fil_holes_,
            fit_rect_=fit_rect_,
            show_msk=False)
        pr_msk = (pr_msk*255).astype('uint8')
        if save_path is not None:
            cv2.imwrite(f'{save_path}/{name_}', pr_msk)

        

In [ ]:
msk_fn = np.random.choice(msk_path.ls())
msk_img=read_img(msk_fn, cv=True, gray=True)
lbl_msk = measure.label(msk_img)
props = measure.regionprops_table(lbl_msk, msk_img, properties=[
    'label', 
    'area', 
    #'bbox',
    'orientation',
    'euler_number',
    'filled_area',
    'solidity',#circle or not
    'equivalent_diameter'])
print(props)

In [ ]:
#| export
def extract_problems(
        pr_img:np.ndarray, # processed image
        img:np.ndarray,  # actual image
        save_path:Union[Path, None]=None,
        save_name:Union[str, None]=None,
        msk:Union[np.ndarray, None]=None, # actual mask
        save_:bool=False,
        img_show_:bool=False,
        pr_img_show_:bool=False,
        msk_show_:bool=False,
        concat_image_path:Union[Path, None]=None,
        concat_image_name:str=None
        ):
    'cut specific parts from the processed image, where model failed'
    hsv = cv2.cvtColor(pr_img, cv2.COLOR_RGB2HSV)
    # searching for blue color, after reading processed image, it seems problem pins are not red, but blue
    # therefore blue color is searched
    mask = cv2.inRange(hsv, (100,90, 255 ), (120,255, 255))
    label_img = measure.label(mask)
    props = measure.regionprops_table(
        label_img,
        img,
        properties=[
            'label',
            'area',
            'bbox',
            'orientation',
            'euler_number',
            'filled_area',
            'solidity',#circle or not
            'equivalent_diameter']
    )
    if save_:
        Path(save_path).mkdir(exist_ok=True, parents=True)
    if concat_image_path is not None:
        Path(concat_image_path).mkdir(exist_ok=True, parents=True)
    for i in range(len(props['bbox-0'])):
        x_min, y_min, x_max, y_max = props['bbox-0'][i],props['bbox-1'][i], props['bbox-2'][i], props['bbox-3'][i]
        if save_:
            cv2.imwrite(f'{save_path}/{save_name}_{i}.png',img[x_min:x_max, y_min:y_max])
        if img_show_:
            show_(img[x_min:x_max, y_min:y_max])
        if pr_img_show_:
            show_(pr_img[x_min:x_max, y_min:y_max])
        if msk_show_:
            show_(msk[x_min:x_max, y_min:y_max])
        if concat_image_path is not None:
            cnct_img = np.concatenate([img[x_min:x_max, y_min:y_max]], axis=0)
            cv2.imwrite(f'{concat_image_path}/{concat_image_name}', cnct_img)

In [ ]:
#| export
def pin_type_accuracy(
    df: pd.DataFrame
):
    "Calculate each pin type accuracy"

    df_fail = df.query('expected_vs_diff != 0')['pin_type'].value_counts(
    ).reset_index().rename(columns={'count': 'pin_type', 'count': 'count'})
    print(f'Total number of pins didnot match: {df_fail["count"].sum()}')

    for i in df_fail['pin_type'].unique():
        total_c = len(df.query('pin_type == @i'))
        fail = df_fail.query('pin_type == @i')['count'].values[0]
        print(f'{i} accuracy: {round((total_c-fail)/total_c*100,5)}%')

    print(f' Addtional pin count {"#"*50}')
    df_fail_a = df.query('expected_vs_diff > 0')['pin_type'].value_counts(
    ).reset_index().rename(columns={'count': 'pin_type', 'count': 'count'})
    for i in df_fail_a['pin_type'].unique():
        fail = df_fail_a.query('pin_type == @i')['count'].values[0]
        print(f'{i} had additional pin count = {fail}')

    print(f' Missing pin count {"#"*50}')
    df_fail_m = df.query('expected_vs_diff < 0')['pin_type'].value_counts(
    ).reset_index().rename(columns={'count': 'pin_type', 'count': 'count'})
    for i in df_fail_m['pin_type'].unique():
        fail = df_fail_m.query('pin_type == @i')['count'].values[0]
        print(f'{i} has missing pin count = {fail}')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('01_mask_accuracy.ipynb')